In [2]:
import os
import numpy as np

import rasterio
import rasterio.features

import fiona
from shapely.geometry import shape, mapping, LineString
from shapely.geometry import shape

from skimage import filters, morphology

# EDIT PATHS
INPUT_TIFF = "data/el_harrach_georef.tif"
OUT_DIR = "output"
os.makedirs(OUT_DIR, exist_ok=True)


# --- 1. Read raster
def read_as_single_band(tiff_path):
    with rasterio.open(tiff_path) as src:
        arr = src.read().astype("float32")
        if arr.shape[0] > 1:
            band = np.mean(arr, axis=0)
        else:
            band = arr[0]
        return band, src.transform, src.crs


band, transform, crs = read_as_single_band(INPUT_TIFF)


# --- 2. Buildings mask
def make_buildings_mask(intensity):
    thresh = filters.threshold_otsu(intensity)
    mask = intensity > thresh
    mask = morphology.binary_closing(mask, selem=morphology.disk(2))
    mask = morphology.binary_opening(mask, selem=morphology.disk(1))
    return mask.astype("uint8")


buildings_mask = make_buildings_mask(band)


# --- 3. Roads mask
def make_roads_mask(intensity):
    from skimage.filters import sobel

    norm = intensity / intensity.max()
    edge = sobel(norm)

    thresh = filters.threshold_otsu(edge)
    mask = edge > thresh

    mask = morphology.binary_closing(mask, selem=morphology.disk(1))
    mask = morphology.binary_opening(mask, selem=morphology.disk(1))
    return mask.astype("uint8")


roads_mask = make_roads_mask(band)


# --- 4. Vectorize buildings → polygons
def raster_to_polygons(raster_mask, transform, crs, output_path):
    schema = {"geometry": "Polygon", "properties": {"class": "str"}}

    with fiona.open(
        output_path,
        "w",
        driver="ESRI Shapefile",
        crs=crs,                # ← now just pass rasterio CRS directly
        schema=schema,
    ) as dst:
        shapes = rasterio.features.shapes(raster_mask, mask=raster_mask, transform=transform)
        for polygon, value in shapes:
            if value == 0:
                continue
            geom = shape(polygon)
            simple = geom.simplify(0.5, preserve_topology=True)
            dst.write(
                {
                    "geometry": mapping(simple),
                    "properties": {"class": "building"},
                }
            )


raster_to_polygons(
    buildings_mask,
    transform,
    crs,
    os.path.join(OUT_DIR, "buildings.shp")
)


# --- 5. Vectorize roads → lines
def raster_to_lines(raster_mask, transform, crs, output_path):
    from skimage.morphology import skeletonize

    skeleton = skeletonize(raster_mask).astype("uint8")
    rows, cols = np.nonzero(skeleton)

    if rows.size == 0:
        print("No nonzero pixels for roads; skipping line export.")
        return

    coords = [rasterio.transform.xy(transform, row, col, offset="center") for row, col in zip(rows, cols)]

    if len(coords) > 1:
        line = LineString(coords)
        line = line.simplify(0.5, preserve_topology=True)

        schema = {"geometry": "LineString", "properties": {}}

        with fiona.open(
            output_path,
            "w",
            driver="ESRI Shapefile",
            crs=crs,                # ← same here
            schema=schema,
        ) as dst:
            dst.write(
                {
                    "geometry": mapping(line),
                    "properties": {},
                }
            )


raster_to_lines(
    roads_mask,
    transform,
    crs,
    os.path.join(OUT_DIR, "roads.shp")
)

print("✓ All vectorization done. Check output folder:", OUT_DIR)

/tmp/ipykernel_220162/1166838661.py:37: FutureWarning: `binary_closing` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.closing` instead.
  mask = morphology.binary_closing(mask, selem=morphology.disk(2))


TypeError: binary_closing() got an unexpected keyword argument 'selem'